# __call__ vs __init__

在 Python 中，`__init__` 和 `__call__` 是两个非常重要的特殊方法（魔术方法），它们分别控制着**对象的创建初始化**和**对象的调用行为**。

简单来说：
*   **`__init__`** 决定了**如何创建一个对象**（当你使用 `MyClass()` 时触发）。
*   **`__call__`** 决定了**如何把一个对象当作函数来使用**（当你使用 `obj()` 时触发）。

| 对比维度 | `__init__` | `__call__` |
| :--- | :--- | :--- |
| **触发语法** | `obj = MyClass(args)` | `result = obj(args)` |
| **调用主体** | **类** (Class) | **实例对象** (Instance) |
| **核心目的** | 初始化对象状态，绑定初始属性 | 定义对象被当作函数调用时的具体行为 |
| **执行频率** | 对象创建时执行 **1 次** | 每次把对象当函数用时执行，可执行 **N 次** |

例如： 下面的代码中，
```python
a = Linear(3,4) # 这里调用的就是`init`, 基于类初始化一个实例
a(xxx)          # 这里调用的就是`call`, 直接调用实例
```

# pytorch化的实现

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


**尝试训练一个更深的神经网络**

In [4]:
# # pytorch的标准module实现中并没有 self.out，这里是为了后续可视化容易显示变量结果，因此对于每个类，都给了个 self.out 属性
class Linear:
  """https://docs.pytorch.org/docs/2.13/generated/torch.nn.Linear.html
  """
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  """https://docs.pytorch.org/docs/2.13/generated/torch.nn.BatchNorm1d.html
  """
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True                                               # 并没有体现在初始化的参数中，是后续实例化后直接调用修改
    # https://github.com/pytorch/pytorch/blob/v2.13.0/torch/nn/modules/batchnorm.py#L187 pytorch里的实现，也确实有training这个bool值
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)                                 # 这里用的是方差，不是标准差了，完全和论文保持一致了       
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance

      # update the buffers
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta                         
   
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]  # 对应pytorch里BN层的weight和bias，不会返回running_mean和running_var，后两个是滑动平均算的，不需要依靠梯度下降算法和反向传播进行计算

class Tanh:
  """https://docs.pytorch.org/docs/2.13/generated/torch.nn.Tanh.html
  """
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

In [5]:
a = Linear(10, 20)
a.weight.shape[0]

10

In [7]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size),
]
with torch.no_grad():
  # last layer: make less confident  
  # logits = h @ W2 + b2  最后一层线性层的结果会被作为logits(之前就是希望logits全部为0/全部为一个相同的数字，这样初始化时候输出的结果对每个类别都是公平的)，所以这里会对W2缩小一点，防止softmax部分过于自信  
  # 详见 4_MLP2_makemore.ipynb中 随机初始化的坏处 前面一个cell的描述，及这个部分的内容
  layers[-1].weight *= 0.1
  # all other layers: apply gain
  # 其他所有层，由于用的是tanh() 所以对应kaiming init里的 5/3 
  # https://docs.pytorch.org/docs/2.12/nn.init.html#torch.nn.init.kaiming_normal_
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 5/3
      # layer.weight *= 5/3/(layer.weight.shape[0]**0.5) 
        
# layers = [
#   Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
# ]
# with torch.no_grad():
#   # last layer: make less confident
#   layers[-1].gamma *= 0.1
#   #layers[-1].weight *= 0.1
#   # all other layers: apply gain
#   for layer in layers[:-1]:
#     if isinstance(layer, Linear):
#       layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

46497


In [ ]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization